# 04 - Modeling: Pelatihan LDA Final (K=8)

## Tujuan
Melatih model Latent Dirichlet Allocation dengan konfigurasi optimal dan
menyimpan seluruh artefak model.

## Asal (Input)
`data/intermediate/` - korpus bersih 285 dokumen hasil Notebook 03

## Keluaran (Output)
- `outputs/model/lda_model.gensim` - model LDA biner (K=8)
- `outputs/model/topic_distribution.csv` - topik dominan per dokumen
- `outputs/model/topic_labels.csv` / `.json` - label 8 topik
- `outputs/model/evaluation_metrics.csv` - metrik model
- `model/lda_visualization.html` - visualisasi pyLDAvis

## Konfigurasi Final
| Parameter | Nilai |
|:---|:---|
| Jumlah topik (K) | 8 |
| Alpha | asymmetric |
| Eta | auto |
| Passes | 30 |
| Iterations | 500 |
| Random seed | 42 |
| Metode inferensi | Online Variational Bayes (bukan Gibbs Sampling) |

In [1]:
import sys, io as _io
from pathlib import Path
_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(_ROOT))
try:
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')
except Exception:
    pass

import pandas as pd, numpy as np
from src import config as C
C.ensure_dirs()
print(C.describe())

KONFIGURASI PROYEK
  Repo          : C:\Users\HAJIWAHYU\Downloads\lda-topik-skripsi-si-uinrf-main
  Dokumen akhir : 285 (2021, 2022, 2023, 2024, 2025)
  Vocabulary    : 1849
  Model LDA     : K=8, alpha=asymmetric, eta=auto, passes=30, iterations=500, seed=42
  Inferensi     : Online Variational Bayes (bukan Gibbs Sampling)
  C_V resmi     : 0.4256
  Cutoff data   : 19 September 2026

## Catatan Metodologis Penting

`modeling.py` melakukan `train_test_split(80/20)`: model dilatih pada
228 dokumen, lalu diprediksi ke seluruh 285. Dokumen di hold-out (20%)
belum pernah dilihat model sehingga theta-nya diffuse.

Akibatnya, distribusi di `outputs/model/topic_distribution.csv` bisa berbeda 1-2
dokumen dari distribusi final. **Distribusi final yang dipakai dashboard
dan skripsi dihitung dengan inferensi penuh** - lihat Notebook 08.

## Langkah 1 - Validasi Fingerprint Label

In [2]:
from src.topic_labels import validate_labels
validate_labels(str(C.LDA_MODEL), str(C.LABELS_JSON))
print('[OK] Fingerprint label cocok dengan model')

[OK] Fingerprint label cocok dengan model

## Langkah 2 - Latih Model LDA Final

Sel berikut melatih ulang model LDA dengan konfigurasi final.
Butuh sekitar 5 menit (30 passes x 500 iterasi).

Untuk sekadar **memverifikasi** model yang sudah ada, lewati sel ini dan
langsung ke Langkah 3. Set `RETRAIN = False` agar tidakMexecute.

In [ ]:
RETRAIN = False   # set True untuk melatih ulang dari nol

if RETRAIN:
    from src import modeling
    modeling.main()
else:
    print('[SKIP] Melewati pelatihan ulang - menggunakan model yang tersimpan')
    print('       (set RETRAIN = True jika ingin dilatih dari nol)')

[SKIP] Melewati pelatihan ulang - menggunakan model yang tersimpan
       (set RETRAIN = True jika ingin dilatih dari nol)

## Langkah 3 - Verifikasi Model Tersimpan

In [4]:
from gensim.models import LdaModel
from gensim import corpora

m = LdaModel.load(str(C.LDA_MODEL))
d = corpora.Dictionary.load(str(C.DICTIONARY_PATH))
print(f'K              : {m.num_topics}')
print(f'Vocabulary     : {len(d)}')
print(f'Passes         : {m.passes}')
print(f'Iterations     : {m.iterations}')
print(f'Alpha (asymm.) : {m.alpha[:3]}...')
print(f'Eta (auto)     : {m.eta[:3]}...')

K              : 8
Vocabulary     : 1849
Passes         : 30
Iterations     : 500
Alpha (asymm.) : [0.02546204 0.02208641 0.02566225]...
Eta (auto)     : [0.5728905  0.15731333 0.13542402]...

## Langkah 4 - Hitung Coherence Score C_V (Protokol Resmi)

In [5]:
import ast
from gensim.models import CoherenceModel

prep = pd.read_csv(C.PREPROCESSED_CSV)
texts = prep['tokens'].apply(ast.literal_eval).tolist()
cv = CoherenceModel(model=m, texts=texts, dictionary=d, coherence='c_v', processes=1).get_coherence()

print(f'C_V terhitung : {cv:.4f}')
print(f'C_V resmi     : {C.COHERENCE_CV_OFFICIAL}')
print('Status        :', 'SESHUI' if abs(cv - C.COHERENCE_CV_OFFICIAL) < 0.0001 else 'BERBEDA')

C_V terhitung : 0.4256
C_V resmi     : 0.4256
Status        : SESHUI

## Langkah 5 - Kata Kunci Setiap Topik

In [6]:
for t in range(m.num_topics):
    words = [w for w, _ in m.show_topic(t, topn=8)]
    print(f'  Topik {t+1}: ' + ', '.join(words))

  Topik 1: variabel, teknologi, model, kualitas, tahu, sukses, nikah, monitoring
  Topik 2: pegawai, publik, extreme_programming, administrasi, pengaruh, variabel, produk, admin
  Topik 3: model, klasifikasi, siakad, kualitas, kategori, identifikasi, teknologi, kualitas_layan
  Topik 4: puas_guna, puas, eucs, ease_use, model_end, ancam, computing_satisfaction, end_user
  Topik 5: model, risiko, teknologi, jual, tahu, usability, indonesia, manajemen
  Topik 6: kualitas, bimbing, variabel, manajemen, learning, karakteristik, tahu, sewa
  Topik 7: extreme_programming, kelola_data, manajemen, lapor, uang, model, kasus, didik
  Topik 8: bimbing_konseling, siswa, langgan, usaha, aman, audit, lapor, tahu

## Ringkasan Fase 04 - Modeling

| Metrik | Nilai |
|:---|---:|
| Jumlah topik (K) | 8 |
| Coherence C_V | **0,4256** |
| Vocabulary | 1.849 |

Model final siap digunakan pada tahap Evaluation (Notebook 05) dan
Build Final Artifacts (Notebook 08).

## Peta Artefak

| Artefak | Dibuat oleh | Consumed by |
|:---|:---|:---|
| `outputs/model/lda_model.gensim` | Notebook 04 | Notebook 05, 08, app |
| `outputs/model/topic_distribution.csv` | Notebook 04 | Referensi perbandingan |
| `outputs/model/topic_labels.csv` / `.json` | Notebook 04 | Notebook 07, app |
| `outputs/model/evaluation_metrics.csv` | Notebook 04 | Dokumentasi |